# Importing Packages

In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt # visualization
import os

# Data Collection

In [5]:
FILEPATH='/kaggle/input/nfl-big-data-bowl-2025/'

### Capture datasets

In [6]:
players_df = pd.read_csv(FILEPATH + 'players.csv')
plays_df = pd.read_csv(FILEPATH + 'plays.csv')
games_df = pd.read_csv(FILEPATH + 'games.csv')
tracking_week_1_df = pd.read_csv(FILEPATH + 'tracking_week_1.csv')
tracking_week_2_df = pd.read_csv(FILEPATH + 'tracking_week_2.csv')
tracking_week_3_df = pd.read_csv(FILEPATH + 'tracking_week_3.csv')
tracking_week_4_df = pd.read_csv(FILEPATH + 'tracking_week_4.csv')
tracking_week_5_df = pd.read_csv(FILEPATH + 'tracking_week_5.csv')
tracking_week_6_df = pd.read_csv(FILEPATH + 'tracking_week_6.csv')
tracking_week_7_df = pd.read_csv(FILEPATH + 'tracking_week_7.csv')
tracking_week_8_df = pd.read_csv(FILEPATH + 'tracking_week_8.csv')
tracking_week_9_df = pd.read_csv(FILEPATH + 'tracking_week_9.csv')

In [7]:
tracking_week_1_df.head()

,gameId,playId,nflId,displayName,frameId,frameType,time,jerseyNumber,club,playDirection,x,y,s,a,dis,o,dir,event
0,2022091200,64,35459.0,Kareem Jackson,1,BEFORE_SNAP,2022-09-13 00:16:03.5,22.0,DEN,right,51.06,28.55,0.72,0.37,0.07,246.17,68.34,huddle_break_offense
1,2022091200,64,35459.0,Kareem Jackson,2,BEFORE_SNAP,2022-09-13 00:16:03.6,22.0,DEN,right,51.13,28.57,0.71,0.36,0.07,245.41,71.21,NaN
2,2022091200,64,35459.0,Kareem Jackson,3,BEFORE_SNAP,2022-09-13 00:16:03.7,22.0,DEN,right,51.20,28.59,0.69,0.23,0.07,244.45,69.90,NaN
3,2022091200,64,35459.0,Kareem Jackson,4,BEFORE_SNAP,2022-09-13 00:16:03.8,22.0,DEN,right,51.26,28.62,0.67,0.22,0.07,244.45,67.98,NaN
4,2022091200,64,35459.0,Kareem Jackson,5,BEFORE_SNAP,2022-09-13 00:16:03.9,22.0,DEN,right,51.32,28.65,0.65,0.34,0.07,245.74,62.83,NaN


In [9]:
games_df.head()

,gameId,season,week,gameDate,gameTimeEastern,homeTeamAbbr,visitorTeamAbbr,homeFinalScore,visitorFinalScore
0,2022090800,2022,1,9/8/2022,20:20:00,LA,BUF,10,31
1,2022091100,2022,1,9/11/2022,13:00:00,ATL,NO,26,27
2,2022091101,2022,1,9/11/2022,13:00:00,CAR,CLE,24,26
3,2022091102,2022,1,9/11/2022,13:00:00,CHI,SF,19,10
4,2022091103,2022,1,9/11/2022,13:00:00,CIN,PIT,20,23


`frameType` has to be to numeric values;  
`[0 = BEFORE_SNAP,  
  1 = AFTER_SNAP,  
  2 = SNAP]`

In [10]:
tracking_week_1_df["frameType"].value_counts()

frameType
BEFORE_SNAP    4647564
AFTER_SNAP     2412240
SNAP             44896
Name: count, dtype: int64

`playDirection` has to be encoded into binary values;  
`[0 = left, 
  1 = right]`

In [11]:
tracking_week_1_df["playDirection"].value_counts()

playDirection
left     3556260
right    3548440
Name: count, dtype: int64

`club` will be one-hot encoded to create 66 new features (2 features per club defense/offense)

In [12]:
tracking_week_1_df["club"].value_counts() 

club
football    308900
IND         269379
HOU         269379
PHI         248380
DET         248380
CIN         247115
PIT         247115
ATL         225632
NO          225632
CAR         214720
CLE         214720
LAC         213422
LV          213422
SEA         208505
DEN         208505
JAX         206877
WAS         206877
ARI         206734
KC          206734
BUF         199804
LA          199804
SF          198627
CHI         198627
GB          198352
MIN         198352
NYJ         193919
BAL         193919
NYG         191147
TEN         191147
DAL         188298
TB          188298
NE          186989
MIA         186989
Name: count, dtype: int64

In [13]:
club_keys = tracking_week_1_df["club"].unique()

print(f'{club_keys}\n{len(club_keys)} teams total')

['DEN' 'SEA' 'football' 'TB' 'DAL' 'TEN' 'NYG' 'MIN' 'GB' 'LV' 'LAC' 'KC'
 'ARI' 'JAX' 'WAS' 'NYJ' 'BAL' 'MIA' 'NE' 'IND' 'HOU' 'PHI' 'DET' 'CIN'
 'PIT' 'SF' 'CHI' 'CLE' 'CAR' 'NO' 'ATL' 'BUF' 'LA']
33 teams total


`event` needs to be considered as there are some NaaN values

In [14]:
tracking_week_1_df["event"].value_counts()

event
ball_snap                    44804
line_set                     44252
huddle_break_offense         34477
first_contact                28865
tackle                       27715
pass_forward                 24794
pass_arrived                 18791
man_in_motion                15824
handoff                      15341
pass_outcome_caught          15249
play_action                   8188
pass_outcome_incomplete       8142
shift                         6095
out_of_bounds                 4968
run                           2507
qb_sack                       1495
touchdown                     1104
fumble                        1081
dropped_pass                   920
pass_tipped                    759
fumble_offense_recovered       598
pass_outcome_interception      598
pass_outcome_touchdown         506
play_submit                    483
qb_kneel                       460
fumble_defense_recovered       414
qb_slide                       253
qb_strip_sack                  230
qb_spike      

# Data Cleaning